In [2]:

import os
import pandas as pd
import numpy as np
import scanpy as sc
import scipy.sparse as sp  # 可以从这个命名空间中直接使用 csr_matrix 和 issparse
from cmapPy.pandasGEXpress.parse_gctx import parse
import rdkit
from rdkit.Chem import AllChem
from rdkit import Chem
from tqdm import tqdm
import random
import torch 
from torch.utils.data import Dataset

import h5py


import warnings
warnings.filterwarnings('ignore')

def create_sample_mapping(filtered_info_cp, info_vehicle):
    """we want to map each sample_id in filtered_info_cp to a random sample_id in info_vehicle with the same det_plate
        det plate have plate info and cell line info, so we can use it to match the cell line"""
    vehicle_grouped = info_vehicle.groupby('det_plate')
    result_dict = {}
    
    for _, row in filtered_info_cp.iterrows():
        det_plate = row['det_plate']
        sample_id = row['sample_id']
        
        if det_plate in vehicle_grouped.groups:
            matching_rows = vehicle_grouped.get_group(det_plate)
            random_row = matching_rows.sample(n=1).iloc[0]
            result_dict[sample_id] = random_row['sample_id']
    
    return result_dict



#load datas
print("Loading data...")
info = pd.read_csv('~/cbfp_vae/dataprocess/data/instinfo_beta.txt', sep='\t',header=0,low_memory=False)
info_qp = info[info['qc_pass'] == 1]
info_cp = info_qp[info_qp['pert_type'] == 'trt_cp'] #get the chemical perturbation only
info_vehicle = info_qp[info_qp['pert_type'] == 'ctl_vehicle']
cell_info = pd.read_csv('~/cbfp_vae/dataprocess/data/cellinfo_beta.txt', sep='\t',header=0,)# get cell info

#wcnm comp info have repeat pert id because some compound have multiple target but we only need smiles
# so we drop duplicates here
comp_info = pd.read_csv('~/cbfp_vae/dataprocess/data/compoundinfo_beta.txt', sep='\t',header=0)# get compound info
print("Total compounds:", comp_info.shape[0])
comp_info_unique = comp_info.drop_duplicates(subset=['pert_id'], keep='first')

comp_info_unique_clean = comp_info_unique.dropna(subset=['canonical_smiles'])
print("Unique compounds:", comp_info_unique_clean.shape[0])
gen_info = pd.read_csv('~/cbfp_vae/dataprocess/data/geneinfo_beta.txt', sep='\t',header=0,)# get gene info
my_ds = parse("~/cbfp_vae/dataprocess/data/level3_beta_trt_cp_n1805898x12328.gctx") # trt
cl_ds = parse("~/cbfp_vae/dataprocess/data/level3_beta_ctl_n188708x12328.gctx") # ctl

print("Data loaded!")

print("Merging compound and cell info...")


original_rows = len(info_cp)
print(f"oringinal info_cp: {original_rows}")

info_cp = info_cp.merge(comp_info_unique_clean, how='left', on='pert_id')
info_cp = info_cp.merge(cell_info, how='left', on='cell_iname')

print("Compound and cell info merged!")
print(f'checking {info_cp.shape[0]} samples and {info_cp.shape[1]} features in total')

filtered_info_cp = info_cp[(info_cp['pert_time'] == 24) & (info_cp['pert_dose'] == 10)]

##remove invalid smiles
filtered_info_cp = filtered_info_cp.dropna(subset=['canonical_smiles'])
filtered_info_cp = filtered_info_cp[filtered_info_cp['canonical_smiles'] != 'restricted']
vc = filtered_info_cp["pert_id"].value_counts()
keep_ids = vc[vc >= 10].index
filtered_info_cp = filtered_info_cp[filtered_info_cp["pert_id"].isin(keep_ids)].copy()
print(filtered_info_cp.shape)
sample_mapping = create_sample_mapping(filtered_info_cp, info_vehicle) 
print(f"we get {len(sample_mapping)} pairs of samples")

sample_to_smiles = dict(zip(filtered_info_cp['sample_id'], filtered_info_cp['canonical_smiles']))

canonical_smiles = [sample_to_smiles[sample_id] for sample_id in sample_mapping.keys()]
print("Encoding compound SMILES to fingerprints using Signaturizer...")


gene_info_ctl = [my_ds.data_df[i] for i in sample_mapping.keys()]
gene_info_trt = [
    (my_ds.data_df[i] if i in my_ds.data_df.columns 
    else cl_ds.data_df[i].tolist())
    for i in sample_mapping.values()
    ]
 

print("Gene expression data prepared.")




In [ ]:
required_df = filtered_info_cp[(filtered_info_cp['cell_mfc_name'] == 'PC3') & (filtered_info_cp['cmap_name_x'] == 'pyrvinium-pamoate')] 
required_list = required_df['sample_id'].tolist()
required_list

In [ ]:
sample_mapping

In [ ]:


keys = list(sample_mapping.keys())
cell_idx = []
for item in required_list:
    if item in keys:
        cell_idx.append(keys.index(item))
        print(f"{item} -> idx: {keys.index(item)}")
    else:
        print(f"{item} -> 不在字典的 key 中")

cell_idx

In [ ]:
import pandas as pd

# ── 1. 读取数据 ──────────────────────────────────────────────────────────────
df = pd.read_csv("../PC3attention_condvae_Mi_mean_traindata_all_splits.csv")          # 替换为你的实际文件名
gene_df = pd.read_csv("../dataprocess/data/landmark_genes.csv") # 基因名称文件

# ── 2. 整理基因名称 ──────────────────────────────────────────────────────────
# 假设 gene_landmark.csv 中有一列存放基因名，取出为列表
# 如果列名不是 "gene_name"，请替换为实际列名
gene_names = gene_df.iloc[:, 0].tolist()   # 默认取第一列，共 978 个基因名

# ── 3. 设置 df 的列名 ────────────────────────────────────────────────────────
# df 第一列：无关列，第二列：idx，后面 978 列：基因数据
idx_col   = df.columns[1]                  # 第二列列名（idx）
data_cols = df.columns[2:]                 # 后 978 列

# 用基因名重命名数据列（顺序需与 gene_landmark.csv 一致）
df = df.rename(columns=dict(zip(data_cols, gene_names)))
filtered_df = df[df[idx_col].isin(cell_idx)]

print(f"命中行数: {len(filtered_df)} / {len(cell_idx)}")

# ── 5. 计算每个基因的平均值 ──────────────────────────────────────────────────
gene_means = filtered_df[gene_names].mean()

# ── 6. 排序并取前 50 个基因 ──────────────────────────────────────────────────
top50 = (
    gene_means
    .sort_values(ascending=False)  # 从大到小排序，改为 True 则从小到大
    .head(50)
    .reset_index()
)
top50.columns = ["gene", "mean_expression"]
top50.index += 1  # rank 从 1 开始

print(top50)

# 可选：保存结果
# top50.to_csv("top50_genes.csv", index_label="rank")


In [ ]:
import numpy as np
import random
import os
from collections import defaultdict

# ==============================
# INPUT: your sample_mapping
# ==============================
# sample_mapping: dict
# key = trt sample_id
# value = ctl sample_id

# 示例（实际用你自己的）
# sample_mapping = {...}


# ==============================
# STEP 1: 构建 sample_id → index
# ==============================
sample_ids = list(sample_mapping.keys())

id_to_idx = {sid: i for i, sid in enumerate(sample_ids)}

print(f"Total samples: {len(sample_ids)}")


# ==============================
# STEP 2: 按 cell 分组（存 idx）
# ==============================
cell_groups = defaultdict(list)

for sid in sample_ids:
    try:
        cell = sid.split('_')[1]
    except Exception as e:
        raise ValueError(f"Bad sample_id format: {sid}")

    idx = id_to_idx[sid]
    cell_groups[cell].append(idx)

print(f"Total cells: {len(cell_groups)}")


# ==============================
# STEP 3: 选 Top5 cell
# ==============================
top5_cells = sorted(cell_groups.keys(), key=lambda x: len(cell_groups[x]), reverse=True)[:5]

print("Top5 cells:")
for c in top5_cells:
    print(f"  {c}: {len(cell_groups[c])}")


# ==============================
# STEP 4: split + 保存 idx
# ==============================
save_dir = "splits_idx"
os.makedirs(save_dir, exist_ok=True)

for cell in top5_cells:
    idxs = cell_groups[cell]
    random.shuffle(idxs)

    n = len(idxs)
    n_train = int(0.8 * n)
    n_val = int(0.1 * n)

    train_idx = idxs[:n_train]
    val_idx = idxs[n_train:n_train + n_val]
    test_idx = idxs[n_train + n_val:]

    # 保存
    np.save(f"{save_dir}/{cell}_train_idx.npy", np.array(train_idx))
    np.save(f"{save_dir}/{cell}_val_idx.npy", np.array(val_idx))
    np.save(f"{save_dir}/{cell}_test_idx.npy", np.array(test_idx))

    # ==============================
    # STEP 5: 打印检查
    # ==============================
    print(f"\n{cell}:")
    print(f"  total: {n}")
    print(f"  train: {len(train_idx)}")
    print(f"  val:   {len(val_idx)}")
    print(f"  test:  {len(test_idx)}")

    # 无重叠检查
    assert len(set(train_idx) & set(val_idx)) == 0
    assert len(set(train_idx) & set(test_idx)) == 0
    assert len(set(val_idx) & set(test_idx)) == 0


# ==============================
# STEP 6: 使用示例
# ==============================
# 加载 idx
# train_idx = np.load("splits_idx/PC3_train_idx.npy")

# 用 iloc 取表达矩阵
# expr = my_ds.data_df.iloc[:, train_idx]

print("\nDone.")

In [ ]:
import numpy as np

# 读一个文件（记得 allow_pickle=True）
train_ids = np.load("splits_idx/A549_train_idx.npy", allow_pickle=True)

print(type(train_ids))
print(len(train_ids))
print(train_ids[:5])